<a href="https://colab.research.google.com/github/Marconi-Lab/Swahili_ASR_Model/blob/main/Fine_tuning_(a_pretrained_model)_for_Swahili_ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook, we aim to take a pre-trained model from hugging face using [Whisper](https://huggingface.co/openai/whisper-large-v2). The most recent pre-trained model for ASR from OpenAI. And its also accompanied by an [article on arxiv](https://arxiv.org/pdf/2212.04356.pdf) published in december 2022. 

(we will also try using [Wav2Vec2](https://ai.facebook.com/blog/wav2vec-20-learning-the-structure-of-speech-from-raw-audio/) and  [XLSR-Wav2Vec2](https://ai.facebook.com/blog/-xlm-r-state-of-the-art-cross-lingual-understanding-through-self-supervision/) :
[Wav2Vec2-XLS-R-300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m)
, [Wav2Vec2-XLS-R-1B](https://huggingface.co/facebook/wav2vec2-xls-r-1b)
and [Wav2Vec2-XLS-R-2B](https://huggingface.co/facebook/wav2vec2-xls-r-2b). )
 
 
and fine-tuning with [swahili data](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0) from mozilla common voice hosted in hugging face dataset platform. 


## Install all the requirements

In [1]:
!nvidia-smi
!pip install datasets
!pip install transformers==4.11.3
!pip install torchaudio==0.10.0+cu113 -f https://download.pytorch.org/whl/cu113/torch_stable.html #Install version 0.10.0 with CUDA support for NVIDIA GPUs.
!pip install jiwer

Fri Feb 10 07:20:57 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 510.47.03    Driver Version: 510.47.03    CUDA Version: 11.6     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   69C    P0    28W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

 We will use notebook_login() function to access token which we then use to authenticate to the Hugging Face Hub and allow us to download datasets,  models, and save our checkpoints during training. The Git Large File Storage (LFS) package will help us upload your model checkpoints:

In [2]:
from huggingface_hub import notebook_login
notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [3]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree       
Reading state information... Done
git-lfs is already the newest version (2.9.2-1).
The following package was automatically installed and is no longer required:
  libnvidia-common-510
Use 'apt autoremove' to remove it.
0 upgraded, 0 newly installed, 0 to remove and 21 not upgraded.


## Data

In this stage, we download the common voice data, and the prepare it for fine-tuning one of the three pre-trained models we mentioned at the beginning.

In [4]:
from datasets import load_dataset

training_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="train")
testing_data = load_dataset("mozilla-foundation/common_voice_11_0", "sw", split="test")

Computing checksums:   8%|8         | 1/12 [00:05<00:55,  5.03s/it]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/5 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 9665it [00:00, 96644.10it/s]
Reading metadata...: 26614it [00:00, 112446.83it/s]


...

Generating other split: 0 examples [00:00, ? examples/s]





Reading metadata...: 0it [00:00, ?it/s]



Reading metadata...: 10380it [00:00, 103794.41it/s]



Reading metadata...: 24111it [00:00, 123503.72it/s]



Reading metadata...: 37518it [00:00, 128324.90it/s]



Reading metadata...: 50920it [00:00, 130567.45it/s]



Reading metadata...: 63977it [00:00, 127254.41it/s]



Reading metadata...: 77331it [00:00, 129352.49it/s]



Reading metadata...: 90280it [00:00, 124175.92it/s]



Reading metadata...: 102742it [00:00, 119575.83it/s]



Reading metadata...: 115427it [00:00, 121730.75it/s]



Reading metadata...: 127647it [00:01, 120739.16it/s]



Reading metadata...: 140104it [00:01, 121871.05it/s]



Reading metadata...: 152316it [00:01, 100402.86it/s]



Reading metadata...: 164635it [00:01, 106308.32it/s]



Reading metadata...: 175795it [00:01, 107118.70it/s]



Reading metadata...: 188357it [00:01, 112248.88it/s]



Reading metadata...: 200136it [00:01, 113818.93it/s]



Reading metadata...: 211743it [00:01, 112204.15it/s]



Reading 

Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 7495it [00:00, 74926.85it/s]
Reading metadata...: 14988it [00:00, 74145.72it/s]
Reading metadata...: 22404it [00:00, 73352.56it/s]
Reading metadata...: 30167it [00:00, 75020.81it/s]
Reading metadata...: 37672it [00:00, 74516.99it/s]
Reading metadata...: 47470it [00:00, 71786.97it/s]


Dataset common_voice_11_0 downloaded and prepared to /root/.cache/huggingface/datasets/mozilla-foundation___common_voice_11_0/sw/11.0.0/2c65b95d99ca879b1b1074ea197b65e0497848fd697fdb0582e0f6b75b6f4da0. Subsequent calls will reuse this data.


In [5]:
training_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 26614
})

In [6]:
testing_data

Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment'],
    num_rows: 10238
})

###  We observe that:

For training data, we have 11 columns : `['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment']`, and `26614` rows. 

For testing data, we have the same number of columns but now `10238` rows.
    

###  Let try exploring the data:

1. Remove the unrequired columns from both training and testing set
2. Then output 10 random sentences from the trainin set

In [7]:
#we only remain with the path, audio and sentence which are the only columns the model will require for training
training_data = training_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
testing_data = testing_data.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

In [8]:
# we only have the path, audio and sentence as we expected
print(training_data)
print(testing_data)

Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 26614
})
Dataset({
    features: ['path', 'audio', 'sentence'],
    num_rows: 10238
})


In [9]:
#we now generate 10 random sentences from our two datasets
import random
import pandas as pd
from IPython.display import display, HTML
from datasets import ClassLabel

#this function will receive a dataset then output 10 random sentences

def display_random_elements(dataset, num_examples=10):

  #we first confirm that the dataset is more than 10 sentences
    if num_examples > len(dataset):
        raise ValueError("Can't pick more elements than there are in the dataset.")

  # returns a list of unique, randomly selected integers from 0 to dataset-1
    random_indices = random.sample(range(len(dataset)), num_examples)

    # converts the picked examples into a Pandas DataFrame and displays
    df = pd.DataFrame(dataset[random_indices])
    display(HTML(df.to_html()))

In [22]:
# lets see any 10 examples on the training set
display_random_elements(training_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Inaonyesha uwezekano wa jinsi vile mtu ana maambukizi katika koo.
1,Maeneo makubwa ya Wilaya ya Chunya ni makavu
2,Pep alitolewa na Real Madrid kwenye michuano
3,Alianza kwa kufanya mafunzo kwa vitendo kwenye Wizara ya Mambo ya Nje
4,hasili haya ni kwa kweli yanahatari
5,Hii ina maana kwamba haziwezi kufanya virusi vyote kuondoka mwilini mwa mtu
6,Kwa ujumla kuna lahaja zipatazo nne
7,Moja Pele Gwiji
8,Kuna aina kuu mbili za wanyama wa jangwani ambao ni ngamia na bakteria
9,Wilaya hii iko katika kaskazini mashariki ya Tanzania.


In [23]:
#lets also see 10 from testing set
display_random_elements(testing_data.remove_columns(["path", "audio"]), num_examples=10)

,sentence
0,Jacob Zuma alisema kuwa Hayati Nelson Mandela atazikwa qunu
1,Hata hivyo aliendelea kuwa na athari kubwa katika siasa ya Tanzania hadi kifo chake.
2,Yeye pamoja na mume wake wana watoto wanne ambao wanaishi nchi ya Kenya.
3,baada ya kukomeshwa kwa utawala wa kibaguzi
4,Bosi kantwaa kisu
5,Kadiri ya Biblia aliishi huko Mesopotamia.
6,Watu wako wauone wakishauona.
7,hata hivyo kidogo zaidi ya nusu ya wakazi ni Wahindu.
8,Wakati wa kujifunza lugha alianza tayari kukusanya hadithi na masimulizi ya Wachagga.
9,Pia inatoa msingi wa vipengele vya mtandao.


Let's extract all distinct letters of the training and test data and build our vocabulary from this set of letters.

In [34]:
# take a batch of sentences
def extract_all_chars(batch):

  # Concatenates all the sentences in the batch into a single string, separating each sentence with a space character
  all_text = " ".join(batch["sentence"])

  # we remove any duplicates from the sentences 
  vocab = list(set(all_text))

  # we then return a list of unique characters
  return {"vocab": [vocab], "all_text": [all_text]}

In [35]:
# we do map this function to both the training and testing data
training_vocabulary = training_data.map(extract_all_chars, batched=True, batch_size=-1, remove_columns= training_data.column_names )
testing_vocabulary = testing_data.map(extract_all_chars, batched=True, batch_size=-1, remove_columns = testing_data.column_names)

  0%|          | 0/1 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

In [18]:
print(training_vocabulary)
print(testing_vocabulary)

Dataset({
    features: ['vocab', 'all_text'],
    num_rows: 1
})
Dataset({
    features: ['vocab', 'all_text'],
    num_rows: 1
})


We will now create a list of all the unique letters found in both the training and test datasets, and then creating a dictionary where each unique letter is assigned a numerical value 

In [21]:
# we create a new list of unique elements from the training and testing vocabulary list
vocabulary_list = list(set(training_vocabulary["vocab"][0]) | set(testing_vocabulary["vocab"][0]))

# then we create a dictionary of the unique elements and their count
vocabulary_dict = {cha: i for i, cha in enumerate(sorted(vocabulary_list))}
vocabulary_dict


{' ': 0,
 '!': 1,
 '"': 2,
 "'": 3,
 '(': 4,
 ')': 5,
 '*': 6,
 ',': 7,
 '-': 8,
 '.': 9,
 '/': 10,
 ':': 11,
 ';': 12,
 '=': 13,
 '?': 14,
 'A': 15,
 'B': 16,
 'C': 17,
 'D': 18,
 'E': 19,
 'F': 20,
 'G': 21,
 'H': 22,
 'I': 23,
 'J': 24,
 'K': 25,
 'L': 26,
 'M': 27,
 'N': 28,
 'O': 29,
 'P': 30,
 'Q': 31,
 'R': 32,
 'S': 33,
 'T': 34,
 'U': 35,
 'V': 36,
 'W': 37,
 'X': 38,
 'Y': 39,
 'Z': 40,
 '`': 41,
 'a': 42,
 'b': 43,
 'c': 44,
 'd': 45,
 'e': 46,
 'f': 47,
 'g': 48,
 'h': 49,
 'i': 50,
 'j': 51,
 'k': 52,
 'l': 53,
 'm': 54,
 'n': 55,
 'o': 56,
 'p': 57,
 'q': 58,
 'r': 59,
 's': 60,
 't': 61,
 'u': 62,
 'v': 63,
 'w': 64,
 'x': 65,
 'y': 66,
 'z': 67,
 '°': 68,
 'µ': 69,
 'Ã': 70,
 'Å': 71,
 'á': 72,
 'â': 73,
 'é': 74,
 'ï': 75,
 'ñ': 76,
 'ó': 77,
 'ö': 78,
 'ø': 79,
 'ú': 80,
 'Š': 81,
 'ū': 82,
 'ː': 83,
 'ụ': 84,
 '‘': 85,
 '’': 86,
 '”': 87,
 '•': 88,
 '…': 89}

We can remove special characters that do not change the pronounciation of words. As we can see above, characters such as ":",".", etc

We also don't want the model to think that "R" and "r" are different. So we can convert all the characters into lower case

In [32]:
import re
chars_to_remove_regex = '[\,\?\.\!\-\;\:\"\“\%\‘\”\?\'\…\•\°\(\)\=\*\/\`\ː\’]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"]).lower()
    return batch

In [33]:
# we map the "remove_special_characters" function on both the trainnng and testing data

training_data = training_data.map(remove_special_characters)
testing_data = testing_data.map(remove_special_characters)


  0%|          | 0/26614 [00:00<?, ?ex/s]

  0%|          | 0/10238 [00:00<?, ?ex/s]

Lets re-run the last block of code to see whether we have removed the special characters and whether all the letters are lower case

In [36]:
# we create a new list of unique elements from the training and testing vocabulary list
vocabulary_list = list(set(training_vocabulary["vocab"][0]) | set(testing_vocabulary["vocab"][0]))

# then we create a dictionary of the unique elements and their count
vocabulary_dict = {cha: i for i, cha in enumerate(sorted(vocabulary_list))}
vocabulary_dict

{' ': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 'µ': 27,
 'á': 28,
 'â': 29,
 'ã': 30,
 'å': 31,
 'é': 32,
 'ï': 33,
 'ñ': 34,
 'ó': 35,
 'ö': 36,
 'ø': 37,
 'ú': 38,
 'š': 39,
 'ū': 40,
 'ụ': 41}

### Audio data
- loading the swahili audio data
- resampling it and preprocessing it in accordance with the training data used to train whisper model

In [12]:
#we use the Audio object from hugging face datasets function
from datasets import Audio

#we resample the training and testing data to 16KHz as that is the sampling rate used to train the model
training_data = training_data.cast_column("audio", Audio(sampling_rate=16000))
testing_data = testing_data.cast_column("audio", Audio(sampling_rate=16000))

In [14]:
#lets see the first columns

print(training_data[0]["audio"])
print(testing_data[0]["audio"])

{'path': '/root/.cache/huggingface/datasets/downloads/extracted/d0c515954317076cb4654c80caae844d8922490cb28ecb53f2df4b52ab7baa52/common_voice_sw_28660554.mp3', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00162212,
       -0.00171909, -0.00184702], dtype=float32), 'sampling_rate': 16000}
{'path': '/root/.cache/huggingface/datasets/downloads/extracted/51be8e6185f5507509f311a8e86535649a659f2e388d898f86a8a45ad84306fd/common_voice_sw_31428161.mp3', 'array': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 'sampling_rate': 16000}
